# Распознаём изображения: исследуем оценки

Этот ноутбук дополняет § 3.1 числовым опытом. Нужен NumPy; скачивать модель не требуется. После него запустите модель на собственных разрешённых фото по листингу статьи.

## Читаем top-3
Измените логиты и проверьте порядок классов. Почему метка вне списка никогда не появится?

In [ ]:
import numpy as np
labels = np.array(["чашка", "принтер", "кот", "книга"])
logits = np.array([2.0, 1.0, 0.0, -1.0])
shifted = logits - logits.max()
prob = np.exp(shifted) / np.exp(shifted).sum()
order = np.argsort(-prob)[:3]
print(list(zip(labels[order], prob[order].round(3))))
assert np.isclose(prob.sum(), 1)

## Проверяем чувствительность к изменению выхода
Измените только второй логит. Сравните лидера и разрыв между первыми двумя оценками.

In [ ]:
changed = logits.copy()
changed[1] += 1.5
p2 = np.exp(changed - changed.max())
p2 /= p2.sum()
print("Было:", labels[np.argmax(prob)], "стало:", labels[np.argmax(p2)])
print("Разрыв:", round(float(np.sort(p2)[-1] - np.sort(p2)[-2]), 3))

## Своя проверка
Запишите top-3 для трёх вариантов фотографии в статье. Впишите наблюдения в таблицу ниже в Markdown-ячейке; код не подставит правильный класс автоматически.

In [ ]:
observations = []  # добавьте словари: файл, top3, что видно на фото
print("Записано наблюдений:", len(observations))

## Опыт с готовой сетью

Подготовьте три собственных снимка кружки: ровный фон, загруженный стол и крупный план. Сначала предположите, на каком снимке ответ будет вернее. Установите зависимости из статьи, замените имена файлов, затем переключите run_model на True. После запуска добавьте к таблице вручную столбец с правильной меткой и объясните каждую ошибку.


In [ ]:
run_model = False
if run_model:
    from pathlib import Path
    import pandas as pd
    from PIL import Image
    import torch
    from transformers import AutoImageProcessor, AutoModelForImageClassification

    files = ["mug_plain.jpg", "mug_busy.jpg", "mug_crop.jpg"]
    assert all(Path(name).is_file() for name in files)
    model_id = "microsoft/resnet-18"
    processor = AutoImageProcessor.from_pretrained(model_id)
    model = AutoModelForImageClassification.from_pretrained(model_id).eval()
    rows = []
    for name in files:
        image = Image.open(name).convert("RGB")
        with torch.inference_mode():
            logits = model(**processor(images=image, return_tensors="pt")).logits[0]
        values, indices = logits.softmax(0).topk(3)
        labels = [model.config.id2label[i] for i in indices.tolist()]
        rows.append({"file": name, "top1": labels[0], "top3": labels,
                     "score_top1": round(values[0].item(), 3)})
    report = pd.DataFrame(rows)
    print(report.to_string(index=False))


## Разбор результата

Запишите для каждого снимка: виден ли предмет целиком, есть ли отвлекающие предметы и находится ли правильный класс в top-3. Сравните две фотографии при неизменных весах. Попробуйте ещё один предмет, которого нет среди классов ImageNet, и объясните, почему модель всё равно печатает знакомую метку.


In [ ]:
if run_model:
    # Заполните вручную после просмотра фото и списка классов.
    report["correct_top1"] = [False, False, False]
    report["correct_top3"] = [False, False, False]
    print(report[["file", "correct_top1", "correct_top3"]])
    print("top-1:", report["correct_top1"].mean(),
          "top-3:", report["correct_top3"].mean())


## Самопроверка




### 1. Какой объект получает метку в обычной классификации изображения?

A. Каждый пиксель.

B. Всё изображение.

C. Только фон.

D. Каждый найденный прямоугольник.




### 2. Что нужно проверить до применения весов ImageNet к фигуркам кружка?

A. Цвет корпуса компьютера.

B. Число фотографий в папке.

C. Список классов модели.

D. Размер монитора.




### 3. Что такое логит?

A. Размер файла.

B. Число до softmax.

C. Порог обрезки.

D. Название класса.




### 4. Чему равна сумма значений после softmax по всем классам?

A. 0.

B. 0,5.

C. 1.

D. Числу пикселей.




### 5. Можно ли назвать первое значение softmax гарантированной вероятностью правильного ответа на новых фото?

A. Да, всегда.

B. Да, если оно больше 0,5.

C. Нет, нужна отдельная проверка.

D. Только при трёх классах.




### 6. Зачем применять AutoImageProcessor из комплекта модели?

A. Чтобы случайно менять веса.

B. Чтобы повторить ожидаемую подготовку входа.

C. Чтобы найти людей на фото.

D. Чтобы удалить фон.




### 7. Что делает model.eval() в примере?

A. Обучает сеть.

B. Меняет классы.

C. Переводит модель в режим оценки.

D. Сохраняет фото.




### 8. Почему сравнение трёх снимков одной вещи полезно?

A. Оно доказывает качество на всех фото.

B. Оно показывает влияние условий съёмки.

C. Оно меняет веса.

D. Оно добавляет классы.




### 9. На фото два предмета. Какой вопрос обычный классификатор не решает?

A. Какая модель загружена.

B. Где расположен каждый предмет.

C. Сколько логитов выдано.

D. Какой индекс лидирует.




### 10. Что сохранить для воспроизводимого опыта?

A. Только лучший ответ.

B. Только название файла.

C. Модель, подготовку, вход и версии библиотек.

D. Только скриншот.




## Подсказки и решения

### Вопрос 1

**Подсказка**


Посмотрите на число входных кадров и число выводимых классов.



**Решение**


**B.** Классификатор выдаёт оценки классов для кадра целиком.



### Вопрос 2

**Подсказка**


Модель выбирает из заранее заданных меток.



**Решение**


**C.** Если нужного класса нет в списке, верное название модель не выдаст.



### Вопрос 3

**Подсказка**


Это выход модели перед нормировкой оценок.



**Решение**


**B.** Логит может быть любым вещественным числом; softmax превращает логиты в доли.



### Вопрос 4

**Подсказка**


Посмотрите на знаменатель формулы.



**Решение**


**C.** Все числители вместе равны знаменателю, поэтому сумма равна единице.



### Вопрос 5

**Подсказка**


Подумайте, проверяли ли модель на похожих размеченных снимках.



**Решение**


**C.** Оценка внутри списка классов сама по себе не подтверждает частоту верных ответов.



### Вопрос 6

**Подсказка**


Модель обучалась на входах определённого формата.



**Решение**


**B.** Процессор задаёт размер, обрезку и нормализацию, совместимые с весами.



### Вопрос 7

**Подсказка**


В этой главе веса не обновляются.



**Решение**


**C.** Режим оценки отключает поведение обучающих слоёв, зависящее от режима.



### Вопрос 8

**Подсказка**


Меняются фон, свет или обрезка, а модель остаётся той же.



**Решение**


**B.** Так можно увидеть чувствительность ответа, но не измерить общее качество.



### Вопрос 9

**Подсказка**


Выход содержит оценки всего кадра.



**Решение**


**B.** Для положения каждого объекта нужна детекция, рассмотренная дальше.



### Вопрос 10

**Подсказка**


Другой запуск должен получить тот же вход и настройки.



**Решение**


**C.** Идентификатор весов, обработка и исходный файл влияют на результат.



